# Notebook: Using NeMo Framework 24.12 for PEFT (LoRA)

## Section 1:  Setting Up PeFT with LoRA using NeMO Framework

This playbook aims to demonstrate how to adapt or customize foundation models to improve performance on specific tasks. We will use NeMo Framework for parameter efficient fine-tuning (PEFT) of Llama3.1-8B-Instruct and deploy it with NVIDIA NIM for LLMs. 

<img src="images/Fine_tuning_1.png" alt="image" width="800"/>


## Table of Contents:
### Section 1
* Download [Llama-8B-Instruct] in HuggingFace Format
* Convert HF Model to NeMo
* Prepare Example Data
* Run the PEFT Fine-Tuning Script
* Evaluation of PEFT Model
* Setup LoRA Model Directory Structure

### Section 2
* Setup NIM for LLMs
* Run NIM locally
* Run PEFT LORA inside a NIM
* Compare simple output of Base Model vs PEFT LORA Model
* Appendix

### References:

- [Overview - NVIDIA Docs](https://docs.nvidia.com/nemo-framework/user-guide/latest/overview.html)
- [Using NEMO for LORA: A description of parameters](https://docs.nvidia.com/nim/large-language-models/latest/peft.html#lora-setup-overview)
- [NVIDIA NeMo Framework PEFT with Llama2 and Mixtral-8x7B](https://docs.nvidia.com/nemo-framework/user-guide/latest/playbooks/nemoframeworkpeft.html#nemo-framework-peft-playbook) 
- [Parameter-Efficient Fine-Tuning](https://docs.nvidia.com/nim/large-language-models/latest/peft.html)

# Import Environment Variables

In [ ]:
# Stop all docker containers (except nginx server)
!scripts/stop_containers.sh

# Check running containers 
!docker ps -a

# Check which GPU you have and how much memory we have to work with.
!nvidia-smi


In [ ]:
# Setup Environment
import os
from dotenv import load_dotenv
# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"Current working dir           : { os.getcwd() }")
print(f"locally hosted model for lab  : {os.getenv('NIM_SERVED_MODEL_NAME')}")



# Download Llama-3.1-8B-Instruct in HuggingFace Format



This step shows how to download weights from Huggingface and convert to NeMo format for finetuning.<br>
 *NOTE: The models should be in your Launchpad environment.*

This is supplementary information showing how to download the model from HuggingFace:

- Make sure you have access to the gated model [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).

    - To request access, click on the [Model Link](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) and submit your application. After the application has been approved, we can then clone the model to get started. It should be an immediate approval.
    
- `Username` is your HuggingFace account and `Password` is your [Huggingface access token](https://huggingface.co/settings/tokens). There are 2 ways to sign in, either in the the terminal with the CLI (most common) or in a notebook.
    - You might have to enter these details twice and this step takes about 5-10 minutes.  In VScode the input will be in the top dropdown bar.
    - This will take about 10 minutes. You may not see any output.

--------------------

### Download Huggingface weights -- Code Example
Use the following commands to login. They will prompt for input of `Username` and `Password`.

```python
from huggingface_hub import snapshot_download

snapshot_download(repo_id="meta-llama/Llama-3.1-8B-Instruct",
      local_dir="Llama-3.1-8B-Instruct",
      local_dir_use_symlinks=False,
      token="YOUR HF TOKEN")

```

---------------------

### Terminal with CLI Method

```bash
pip install huggingface_hub
huggingface-cli login

git lfs install
git clone https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
```

![image](images/Huggingface_model_dl.png)

### Inspect our current Huggingface Llama model directory

NVIDIA's NeMo framework provides tools to convert models from formats like Hugging Face's safetensors to the .nemo format. This conversion facilitates tasks such as supervised fine-tuning (SFT), parameter-efficient fine-tuning (PEFT), and inference within the NeMo ecosystem.  In the next section we will see how to do that.  Lets inspect our current Llama 3 model directory structure.

In [ ]:
!tree ./models/

# Convert HF Model to NeMo

The NeMo container includes modules to support the conversion of Hugging Face (HF) format to NeMo format. To use the NeMo docker container, we will first login to `nvcr.io`.

In [ ]:
import os

# Get the NGC API key from the environment variable
NGC_API_KEY = os.getenv('NGC_API_KEY')
HF_TOKEN    = os.getenv('HF_TOKEN')

# Log in to nvcr.io using the NGC API key
!echo $NGC_API_KEY | docker login nvcr.io -u '$oauthtoken' --password-stdin
!huggingface-cli login --token {HF_TOKEN}

### Example Method to Start a Docker Container (if running outside of notebook) - *Information Only*



Alternatively, If you run `docker login nvcr.io` (as detailed in NVIDIA docs)
you will be prompted. Then you can use `$oauthtoken` as the username and your[ NGC API Key](https://org.ngc.nvidia.com/setup/personal-keys) as the password.  You will only need to do this once and then it will be added to your cache.

We can run the container using the following command. Note that you may have to update which GPU device(s) are available.  This container runs in the background of our notebook.
in terminal run 
```bash

docker run --gpus '"device=0"' --shm-size=2g --net=host --ulimit memlock=-1 --rm -it -v ${PWD}:/workspace -w /workspace -v ${PWD}/results:/results nvcr.io/nvidia/nemo:24.12 bash
```

If we had run the previous command it would have opened a bash shell within the container. Then using that shell, we would convert the Huggingface model into a .nemo model.



### Continue to convert Huggingface model to NeMo format

Since we are in this notebook we can have our commands run inside the container by appending them to the end of the ```docker run ... ``` command.  Next we append our python conversion script to ```docker run```.

The NeMo container comes with a number of scripts and built in tools.  Here is a printout of the available converters as of Nemo 24.12 release.


In [ ]:
!docker run --gpus '"device=0"' --shm-size=2g --net=host \
    --ulimit memlock=-1 --rm -it \
    -v "${PWD}:/workspace" -w /workspace \
    -v "${PWD}/results:/results" \
    nvcr.io/nvidia/nemo:24.12 \
    ls /opt/NeMo/scripts/checkpoint_converters | grep convert

**NOTE:  The follownig command will run in the background and will take a few minutes to complete.**<br>
This script does the safetensors => *.nemo conversion.

In [ ]:
!docker run --gpus '"device=0"' --shm-size=2g --net=host \
    --ulimit memlock=-1 --rm -it -v ${PWD}:/workspace -w /workspace -v ${PWD}/results:/results nvcr.io/nvidia/nemo:24.12 \
        python /opt/NeMo/scripts/checkpoint_converters/convert_llama_hf_to_nemo.py \
            --input_name_or_path=./models/Llama-3.1-8B-Instruct --output_path=./models/llama-8b-instruct.nemo

In [ ]:
!tree models

You may receive a warning that `MegatronGPTModel() does not have field.name:` Please ignore these comments as it will not affect the model conversion.

Since we load the current directory inside the container, all the files created inside the docker can be accessed even after the container is stopped. 


# Prepare Dataset

### Dataset Download & Prep in JSONL format for NeMo

We will use the `PubMedQA` dataset as an example. For more details about the data refer to [PubMedQA: A Dataset for Biomedical Research Question Answering](https://arxiv.org/abs/1909.06146) and [PubMedQA](https://github.com/pubmedqa/pubmedqa).

**Note: The data should already be downloaded in your Launchpad. We include these instructions if you want to test locally in the future**

We can download the PubMedQA dataset from GitHub and run the `split_dataset.py` script in the cloned directory.

In [ ]:
# Already downloaded.  If running locally this only takes 1 to 2 mintues.
import os
if os.path.exists('pubmedqa'):
    print("pubmedqa already downloaded")
else:
    print("downloading pubmedqa")   
    !git clone https://github.com/pubmedqa/pubmedqa.git
    os.chdir('pubmedqa/preprocess')
    !python3 split_dataset.py pqal
    os.chdir('../..')


In [ ]:
!tree pubmedqa/data

After running the `split_dataset.py` script, you will see the `test_set.json` as well as ten different directories which each contains a different train/validation fold inside the `data` directory. 

Here is an example that shows what a single row looks like in the `train_set.json` file inside one fold of PubMedQA train, validation and test splits.

```json
"18251357": {
    "QUESTION": "Does histologic chorioamnionitis correspond to clinical chorioamnionitis?",
    "CONTEXTS": [
        "To evaluate the degree to which histologic chorioamnionitis, a frequent finding in placentas submitted for histopathologic evaluation, correlates with clinical indicators of infection in the mother.",
        "A retrospective review was performed on 52 cases with a histologic diagnosis of acute chorioamnionitis from 2,051 deliveries at University Hospital, Newark, from January 2003 to July 2003. Third-trimester placentas without histologic chorioamnionitis (n = 52) served as controls. Cases and controls were selected sequentially. Maternal medical records were reviewed for indicators of maternal infection.",
        "Histologic chorioamnionitis was significantly associated with the usage of antibiotics (p = 0.0095) and a higher mean white blood cell count (p = 0.018). The presence of 1 or more clinical indicators was significantly associated with the presence of histologic chorioamnionitis (p = 0.019)."
    ],
    "reasoning_required_pred": "yes",
    "reasoning_free_pred": "yes",
    "final_decision": "yes",
    "LONG_ANSWER": "Histologic chorioamnionitis is a reliable indicator of infection whether or not it is clinically apparent."
}
```

We will use the following script to convert the train, validation, and test PubMedQA data into the JSONL format that NeMo requires for PEFT finetuning. This is a data wrangling exercise, see the [preprocess_to_jsonl.py](./scripts/preprocess_to_jsonl.py) to see the details of how we format to JSONL. 

In [ ]:
!python3 scripts/preprocess_to_jsonl.py

# Output written to jsonl format in nim-workshop directory.  Moving to pubmedqa for next steps
!mv *.jsonl pubmedqa
!echo
!echo The following JSONL files will be used for finetuning
!ls pubmedqa/*jsonl

After running above commands, you will see the `train_set.jsonl`, `test_set.jsonl`, `val_set.jsonl` files appear in the `pubmedqa` directory.

The following example shows what the formatting will look like after the script has converted the PubMedQA data into the format that NeMo expects for PEFT:

```json
{
    "input": "QUESTION: Failed IUD insertions in community practice: an under-recognized problem?\nCONTEXT: The data analysis was conducted to describe the rate of unsuccessful copper T380A intrauterine device (IUD) insertions among women using the IUD for emergency contraception (EC) at community family planning clinics in Utah.\n...",
    "output": "yes"
}
```


# Run the PEFT Fine-Tuning Script

The `megatron_gpt_finetuning_config.yaml` file is used to configure the parameters for running the PEFT training jobs in NeMo with P-Tuning and LoRA techniques for model tuning.

Next, we will create a shell script to run the fine-tuning. This script also contains all the environment variables for successful execution. The environment variables specified at the top of the script assume you are at `/workspace`. You might need to change these environment variables, depending on the dataset you are using, model size and required training configurations. You can find some examples of different configurations in this [link](https://docs.nvidia.com/nemo-framework/user-guide/latest/playbooks/nemoframeworkpeft.html#nemo-framework-peft-playbook).



Documentation and more info for the above configurations can be found in [here](https://github.com/NVIDIA/NeMo/blob/main/examples/nlp/language_modeling/conf/megatron_gpt_config.yaml).

In the next cell we will run our [run_peft.sh](./scripts/run_peft.sh) script (click link to see the script).  This will kick off the fine tuning job on our Launchpad H100s.   Please note that this step will take about some time based on the above parameters and dataset.

### Kick off PEFT Fine Tuning
*Note:  The next cell should take about 3 to 5 mintues to complete the fine tuning process. We are trianing for minimal run to create a model. But for further training, you can increase the number of epochs and batch size.*


In [ ]:
# Start Fine Tuning Job
# 3 to 5 minutes of run time

# Clear previous experiment if exists
!sudo rm -rf nemo_experiments/

!docker run --gpus '"device=all"' --shm-size=2g --net=host -e HF_TOKEN=$HF_TOKEN \
    --ulimit memlock=-1 --rm -it -v ${PWD}:/workspace -w /workspace -v ${PWD}/results:/results nvcr.io/nvidia/nemo:24.12 \
        bash scripts/run_peft.sh

In [ ]:
!ls nemo_experiments/megatron_gpt_peft_lora_tuning/checkpoints

The PEFT weights are stored in `nemo_experiments/megatron_gpt_peft_lora_tuning/checkpoints` folder, under the name of `megatron_gpt_peft_lora_tuning.nemo`

------

# Evaluate PEFT model

### Generate predictions and calculate test loss

We run the  script in scripts folder called `eval_peft.sh`. You can change the config values depending on your use case. The output of this script is the loss values of model output, and the predictions of the PEFT model on the chosen dataset. 
*Note: We are using the PubMedQA dataset to showcase evaluation.*  

In [ ]:
# about 12 minutes

import os

if os.path.exists('./data/peft_results_test_pubmedqa_inputs_preds_labels.jsonl'):
    print("Evaluation file already exists, not running evaluation")
    pass
else:
    # Execute code when pubmedqa-peft-accuracy-results.json does not exist
    print("Evaluation file does not exist, running eval")
    !docker run --gpus '"device=0,1"' --shm-size=2g --net=host \
        --ulimit memlock=-1 --rm -it -v ${PWD}:/workspace -w /workspace -v ${PWD}/results:/results nvcr.io/nvidia/nemo:24.12 \
            bash scripts/eval_peft.sh
    # copy the evaluation results to data directory
    !cp ./results/peft_results_test_pubmedqa_inputs_preds_labels.jsonl \
        ./data/peft_results_test_pubmedqa_inputs_preds_labels.jsonl
    
    !echo evaluation complete!

If the file already exists then the script will print "File Exists" otherwise, the output will look like this.

![image](images/Peft_eval_out.png)

The output of the above script is the loss metric values of base model output, and the model predictions stored as `results/peft_results_test_${TEST_NAMES}inputs_preds_labels.jsonl`. We can use these predictions for further evaluation metrics.


We will use the python library `evaluate` to calculate the evaluation metrics. The evaluation metrics include `accuracy`, `f1`, `precision`, `recall`, and `loss`. 

### Accuracy metrics - PubMedQA Dataset
For binary or other numerically representable outputs, we can use accuracy metrics. Since we are using the **PubMedQA** dataset for this workshop, you can follow this accuracy metric to test your PEFT weights. If you are using a summarization dataset, please use the ROUGE and BLEU scores for evaluation (in the APPENDIX).

Hence we can calculate the accuracy metrics for the PubMedQA dataset. We run the following functions that will calculate the accuracy and also print a confustion matrix.

Now we run the evaluation script over the `peft_results_test_pubmedqa_inputs_preds_labels.jsonl`. 

In [ ]:
# PEFT Accuracy - Based on first word returned from chat model and some postprocessing
from utils import evaluate_predictions, plot_confusion_matrix
file_name = "./data/peft_results_test_pubmedqa_inputs_preds_labels.jsonl"
predictions, labels, accuracy = evaluate_predictions(file_name, postprocessing=True)

print(f"Accuracy: {accuracy:.2f}")

plot_confusion_matrix(predictions, labels, classes=["yes", "no", "maybe"])


After the accuracy calculation finishes, you should see output containing the accuracy and and a confusion matrix. The sample scores can be improved by training the model further and hyperparameter tuning. 

### Evaluate the Base Model (optional)

Similarly, to test the base model (without PEFT weights), we will make another run with `eval_base.sh` to save teh metrics from the base model to compare.  Let's follow the same process as above to evaluate the base model.

*Note: This script will take the longest to run.  About 1.5 hours.*

This evaluation will give us the base model's accuracy and Macro-F1 test metrics by querying the model many times. If you watch the print out (later after the course) you can see many of these queries being made.

In [ ]:
# you should not need to run this during the course since we prepopulated the output for you. But if you run on your own, you can run this to get the results
# about 1.5 hours

if os.path.exists('./data/base_results_test_pubmedqa_inputs_preds_labels.jsonl'):
    print("Base evaluation file already exists, not running evaluation")
    pass
else:
    # Execute code when pubmedqa-base-accuracy-results.json does not exist
    print("File does not exist, running eval")
    !docker run --gpus '"device=0"' --shm-size=2g --net=host \
        --ulimit memlock=-1 --rm -it -v ${PWD}:/workspace -w /workspace -v ${PWD}/results:/results nvcr.io/nvidia/nemo:24.12 \
        bash scripts/eval_base.sh

    # copy the base evaluation results to data directory
    !cp ./results/base_results_test_pubmedqa_inputs_preds_labels.jsonl \
        ./data/base_results_test_pubmedqa_inputs_preds_labels.jsonl

    !echo evaluation complete

![image](images/eval_base_queries.png)

The output of the above script is just the loss values of base model output. The model predictions are stored as `results/base_results_test_${TEST_NAMES}inputs_preds_labels.jsonl`. Similar to our PEFT output file, we can use these predictions for further evaluation metrics.


In [ ]:
from utils import evaluate_predictions, plot_confusion_matrix
# Baseline Accuracy - Based on first word returned from chat model and some postprocessing
file_name = "./data/base_results_test_pubmedqa_inputs_preds_labels.jsonl"
predictions, labels, accuracy = evaluate_predictions(file_name, postprocessing=True)

print(f"Accuracy: {accuracy:.2f}")
plot_confusion_matrix(predictions, labels, classes=["yes", "no", "maybe"])

As  you can see, we improved from 53% to 71%.  Even better results can likely be achieved via more data and fine tuning.

------

# Examples: Setup LoRA Model Directory Structure

To load the PEFT model weights in NIM, we need to create a `loras` directory, with the following model directory structure.  This is an example of how it would look if we had used llama3.

```bash
loras
├── llama3-8b-math
│   └── llama3_8b_math.nemo
├── llama3-8b-math-hf
│   ├── adapter_config.json
│   └── adapter_model.bin
├── llama3-8b-squad
│   └── squad.nemo
└── llama3-8b-squad-hf
    ├── adapter_config.json
    └── adapter_model.safetensors
```
The above directory example would load four LoRA models, namely, `llama3-8b-math`, `llama3-8b-math-hf`, `llama3-8b-squad`, and `llama3-8b-squad-hf`. For more information, please refer to our [PEFT documentation](https://docs.nvidia.com/nim/large-language-models/latest/peft.html).

For this example, we trained one LoRA model. We can name it `lora-nemo-fw`, and copy our PEFT .nemo model from checkpoints to the `loras` directory. 

In [ ]:
!mkdir -p loras/lora-nemo-fw
!cp nemo_experiments/megatron_gpt_peft_lora_tuning/checkpoints/megatron_gpt_peft_lora_tuning.nemo loras/lora-nemo-fw/lora-nemo-fw.nemo
!chmod -R 777 loras


In [ ]:
!tree loras

Finally, please verify that your `loras` directory looks like this:

```bash
loras/
└── lora-nemo-fw
    └── lora-nemo-fw.nemo
```



*We will cover steps to load this structure in NIM container in the next section if this notebook.*


--------------

# SECTION 2: 
## Deploying NIM with LoRA 

This section walks through all the steps to deploy the PEFT model with NIM, and run inferences with it.

Table of Contents:
* NGC and Docker Login
* Launch NIM
* Run Inference
* NIM with LoRA

### Launch NIM with base model

Before launching the below NIM, we create a readable and writable cache directory on the host machine:
Please insert your allocated port in this script as the `PORT` variable.

In [ ]:
# Choose a container name and port
import os
os.environ["CONTAINER_NAME"]="base-llama-8b-instruct"
os.environ["PORT"]="8000"

# Print current environment variables
print(f"Container Name: {os.getenv('CONTAINER_NAME')}")
print(f"Port: {os.getenv('PORT')}")
print(f"Image Name: {os.getenv('NIM_IMG')}")
print(f"Local Cache: {os.getenv('LOCAL_NIM_CACHE')}")

Launching an NVIDIA NIM is incredibly easy. The below command will launch a Docker container for the `base-llama-8b-instruct` base model and NVIDIA NIM will automatically choose from the vLLM or TRT-LLM backend and will run the most optimal inference engine for that backend. Please update the GPU device as per availability.

Depending on the availability of container names and ports, you may have to customize the `CONTAINER_NAME` and localhost port value.

In [ ]:
# Start the LLM NIM
# Make sure .cache is owned by the USER.  In our course environment this is "nvidia".  
# If you're running locally, you need to change this to your user.  You can check by running 
# `!id -u`

# model_id and container names can be different but naming them the same
!docker run -d --rm --name=$CONTAINER_NAME \
  --runtime=nvidia \
  --gpus '"device=0"' \
  --shm-size=16GB \
  -e NGC_API_KEY \
  -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
  -p $PORT:8000 \
  $NIM_IMG

Next we check to see if the model is running.  This script will continue to check the status with the `while` function until the model is running.  This could take up to about 5 minutes.

In [ ]:
# Check if the model is running
# Define the URL and model ID to check

from utils import check_model_running
import time
from IPython.display import clear_output

url = 'http://0.0.0.0:8000/v1/models'
model_id = os.getenv("NIM_SERVED_MODEL_NAME")

# Check if the model ID exists in the response
while(not check_model_running(url, model_id)):
    print(f"Model ID '{model_id}' not found - Model still loading. Sleeping for 5 seconds.")
    time.sleep(5)
    clear_output(wait=True)  # Clear the previous output

print(f"Model ID '{model_id}' Is: {check_model_running(url, model_id)}")


Once the NIM has been setup, you will see a message on the terminal `Application startup complete`, along with the port where the endpoint is hosted.

### Run Inference on Base Model


Once the NIM container is up and running, please open a different terminal window to run inference tests on the base model.

**OpenAI Completion Request** 

The Completions endpoint is generally used for `base` models. With the Completions endpoint, prompts are sent as plain strings, and the model produces the most likely text completions subject to the other parameters chosen. To stream the result, set `"stream": true`. Please make sure that the port is mentioned correctly in your curl command.

In [ ]:
!curl -X 'POST' \
    'http://0.0.0.0:8000/v1/completions' \
    -H 'accept: application/json' \
    -H 'Content-Type: application/json' \
    -d '{ \
            "model": "meta/llama-3.1-8b-instruct", \
            "prompt": "Once upon a time", \
            "max_tokens": 64 \
        }' | jq

**OpenAI Chat Completion Request**

The Chat Completions endpoint is generally used for `chat` or `instruct` tuned models that are designed to be used through a conversational approach. With the Chat Completions endpoint, prompts are sent in the form of messages with roles and contents, giving a natural way to keep track of a multi-turn conversation. To stream the result, set `"stream": true`.

In [ ]:
import os
import json 

data = {
    "model": "meta/llama-3.1-8b-instruct",
    "messages": [
        {"role": "user", "content": "Hello! How are you?"},
        {"role": "assistant", "content": "Hi! I am quite well, how can I help you today?"},
        {"role": "user", "content": "Can you write me a song?"},
    ],
    "max_tokens": 100,
    "temperature": 0.5,
    "stream": False
}

cmd = f'curl -X POST "http://0.0.0.0:8000/v1/chat/completions" -H "accept: application/json" -H "Content-Type: application/json" -d \'{json.dumps(data)}\''

print(cmd)
!{cmd}|jq

After testing the above inference calls, we can stop our NIM container with `Ctrl+C` and move onto the next step to deploy NIM with LoRA.


### NIM with LoRA

We previously had stored our PEFT weights in the `loras` directory under the model name `lora-nemo-fw` and `llama3-8b-instruct-lora_vnemo-math-v1`. Now, we can load the PEFT weights along with the base model using the following command:
Please make sure that the port is mentioned correctly in your curl command under the `LORA_PORT` variable.

**Note: Change the directory paths for your parent directory**

In [ ]:
os.environ["LOCAL_PEFT_DIRECTORY"]="./loras"
!chmod -R 777 "{os.getenv("LOCAL_PEFT_DIRECTORY")}"
# add math model to loras directory also
!cp -r models/llama3-8b-instruct-lora_vnemo-math-v1 loras/

os.environ["NIM_PEFT_SOURCE"]="/home/nvs/loras"  # PEFT directory inside the NIM container
os.environ["NIM_PEFT_REFRESH_INTERVAL"]="3600"   # will check NIM_PEFT_SOURCE for newly added models every hour
os.environ["CONTAINER_NAME"]="peft-llama-8b-instruct"


#print env variables for this section
print(f"NIM_IMG: {os.getenv('NIM_IMG')}")
print(f"LOCAL_NIM_CACHE: {os.getenv('LOCAL_NIM_CACHE')}")
print(f"NIM_PEFT_SOURCE: {os.getenv('NIM_PEFT_SOURCE')}")
print(f"NIM_PEFT_REFRESH_INTERVAL: {os.getenv('NIM_PEFT_REFRESH_INTERVAL')}")
print(f"CONTAINER_NAME: {os.getenv('CONTAINER_NAME')}")
print(f"LOCAL_PEFT_DIRECTORY: {os.getenv('LOCAL_PEFT_DIRECTORY')}")

### Run the Docker

In [ ]:
# previously running container
!scripts/stop_containers.sh
# https://docs.nvidia.com/nim/large-language-models/latest/configuration.html#environment-variables

# Start PEFT NIM container
!docker run -d --rm --name=$CONTAINER_NAME \
    --runtime=nvidia \
    --gpus '"device=0"' \
    --shm-size=16GB \
    -e NGC_API_KEY \
    -e NIM_PEFT_SOURCE \
    -e NIM_PEFT_REFRESH_INTERVAL \
    -v $LOCAL_NIM_CACHE:/opt/nim/.cache \
    -v $LOCAL_PEFT_DIRECTORY:$NIM_PEFT_SOURCE \
    -p 8000:8000 \
    $NIM_IMG

Next we check to see if the model is running.  This script will continue to check the status with the `while` function until the model is running.  This could take up to about 5 minutes.

In [ ]:
# Check if the model is running
# Define the URL and model ID to check

from utils import check_model_running
import time
from IPython.display import clear_output

url = 'http://0.0.0.0:8000/v1/models'
model_id = os.getenv("NIM_SERVED_MODEL_NAME")

# Check if the model ID exists in the response
while(not check_model_running(url, model_id)):
    print(f"Model ID '{model_id}' not found - Model still loading. Sleeping for 5 seconds.")
    time.sleep(5)
    clear_output(wait=True)  # Clear the previous output

print(f"Model ID '{model_id}' Is: {check_model_running(url, model_id)}")

### Checking for Local Base and PEFT Models

To see the LORA models we have stored locally, we can use this command. 


In [ ]:
!curl -s http://0.0.0.0:$PORT/v1/models | jq

You should see a base model with the ID `base-llama-8b-instruct` as well as the `lora-nemo-fw` and `llama3-8b-instruct-lora_vnemo-math-v1` ( we address this math model in **3.4**).

Next, we can submit a `completion` or `chat completion` inference request against the base model or any LoRA. You can make inference requests to any and all models returned and listed by the command `/v1/models`. The first time you make an inference request to a LoRA adapter, there may be a loading time, but subsequent requests to that same LoRA adapter will have lower latency, as the weights are streamed from the cache.


### Run a Comparison of Base Model vs LoRA Inference with `lora-nemo-fw`



We will ask the same question to the base Llama 3.1 8B model anbd see that it returns an irrational answer.  But when we ask the same question to the LoRA model trained on the pubmedqa (biomedical) dataset we get very specific and accuarate details.

> How does CRISPR-Cas9 gene editing work?

First, we'll pass this question to the original version of `base-llama-8b-instruct` for a baseline: 

In [ ]:
!curl -X 'POST' \
    'http://0.0.0.0:8000/v1/completions' \
    -H 'accept: application/json' \
    -H 'Content-Type: application/json' \
    -d '{ \
            "model": "meta/llama-3.1-8b-instruct", \
            "prompt": "How does CRISPR-Cas9 gene editing work?", \
            "max_tokens": 250 \
        }' | jq


As a fun side note to cmoare model output we look at the response below from Llama 3.0 8B vs the Llama 3.1 8B above.  The Llama3 8B response is somewhat gibberish, it's not totally off base but the quality of response is not great. 

> " [duplicate]\nI've been hearing a lot about CRISPR-Cas9 gene editing and the potential benefits it holds for medicine and various other fields. However, I still don't fully understand how it works.\nHere is my current understanding of the process:\n1. Scientists create a piece of RNA that is complementary to a specific gene they want to edit.\n2. The RNA is introduced into a cell along with the Cas9 enzyme.\n3. The Cas9 enzyme cuts the target DNA sequence, guided by the RNA molecule.\n4. After the DNA is cut, the cell's natural repair machinery kicks in, and it either repairs the cut DNA or introduces small insertions or deletions (indels) at the cut site.\n\nIs my understanding correct? What are the potential risks associated with CRISPR-Cas9 gene editing, and how might these risks be mitigated?\n\nThe CRISPR-Cas9 system is a bacterial immune system that provides resistance against viruses that infect bacteria. The system consists of two main components: the Cas9 enzyme (CRISPR-associated protein 9) and the CRISPR RNA (crRNA).\n\nCas9 is a nuclease that ...\n\n(beware of potential errors as there's no citation)\n\n...binds to a specific",

Now, let's test inference with the LoRA weights we obtained from PEFT on the `pubmedqa` dataset:

In [ ]:
!curl -X 'POST' \
    'http://0.0.0.0:8000/v1/completions' \
    -H 'accept: application/json' \
    -H 'Content-Type: application/json' \
    -d '{ \
            "model": "lora-nemo-fw", \
            "prompt": "How does CRISPR-Cas9 gene editing work?", \
            "max_tokens": 250 \
        }' | jq

The response from the fine tuned Llama 3.1 8B model (lora-nemo-fw) seems to be much better! And decides to tell us an analogy to help learn.

> "Before answering that question, let's travel back to the 19th Century to get an analogy. In 1891, Mendel's laws of genetics revolutionized our understanding of inheritance. Let's look at his example of wrinkled peas. In the diploid nucleus of a pea plant, one of the two pea-shaped chromosome pairs carries the genetic instructions for wrinkled peas, and the other carries genes for round peas. A pea plant will have equal numbers of both types of peas because the single chromosome carrying the gene for wrinkled peas will pair up during meiosis with either the chromosome for round peas or another chromosome with the gene for wrinkled peas. These pairings are known as \"crossing-over.\" This genetic exchange will create an equal mixture of Mendel's pea phenotypes, called genotypes, in future generations.\n\nIn metaphorical terms, Mendel's peas are the letters stuck onto pieces of hardware with Velcro, such that the peas come off easily and can be reattached elsewhere or left off. In this analogy, the genes"

-----


### Install the NGC CLI if needed

  - **Note:** you may need to install [NGC CLI](https://org.ngc.nvidia.com/setup/installers/cli) in order to pull the LoRA weights from NGC to run the above command and export as a local variale to use it. 
  - You will also need to make sure that the NGC CLI is locally in the same parent folder for sit to run. Next you will need to log into the NGC CLI. Then export PATH if there are still issues,
  - If this is not possible with your permissions, please feel free to skip this step.

 How to setup ngc cli --> https://org.ngc.nvidia.com/setup/installers/cli
 1. get version that matches your system
 2. Make the NGC CLI binary executable and add your current directory to path:
```bash
    !chmod u+x ngc-cli/ngc
    !echo "export PATH=\"\$PATH:$(pwd)/ngc-cli\"" >> ~/.bash_profile && source ~/.bash_profile
```


- Download fine tuned math model<br>
*Note: this is already be in your Launchpad, so we will move it to our loras directory*

Sample ngc download
```bash
  !ngc registry model download-version --dest loras "nim/meta/llama3-8b-instruct-lora:nemo-math-v1"
```

-----

### Optional:       Run LoRA Inference with Math Reasoning using  

`llama3-8b-instruct-lora_vnemo-math-v1`

Lastly, we will finish with an example from outside of the medical dataset. The fine-tuning script we previously used for `lora-nemo-fw.nemo` did not run for the optimal number of steps/epochs in the interest of time for this workshop.  We want to demonstrate the effectiveness of creating a LoRA model that has longer training for math reasoning. 

These new weights should already be in your Launchpad directory.  But we will show the process for future reference of how to download.
First download the set of LoRA math weights from NGC that has been fully optimized for math and is fine-tuned (LoRA) on OpenAI's [Grade School Math dataset](https://github.com/openai/grade-school-math).



### After that is installed and setup.  We can check out LORAS folder structure so the NIM knows where to look for the fine tuned models.

Please verify that your `loras` directory looks like this:

```bash
loras/
├── llama3-8b-instruct-lora_vnemo-math-v1
│   ├── LICENSE.txt
│   ├── llama3_8b_math.nemo
│   └── NOTICE.txt
└── lora-nemo-fw
    └── lora-nemo-fw.nemo

In [ ]:
!tree loras

### PEFT Comparison
To see the improvements with PEFT, we run another comparison onf the Base model vs LoRA model:

 Ed has 2 dogs, 3 cats and twice as many fish as cats and dogs combined. How many pets does Ed have in total?

1. First, we'll ask the base version of `base-llama-8b-instruct` to solve this math problem:

In [ ]:
!curl -X 'POST' \
  'http://0.0.0.0:8000/v1/completions' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{ \
        "model": "meta/llama-3.1-8b-instruct", \
        "prompt": "Ed has 2 dogs, 3 cats and twice as many fish as cats and dogs combined. How many pets does Ed have in total?", \
        "max_tokens": 256 \
      }' | jq

You should get a response similar to below, where the LLM struggles to explain its reasoning and ultimately returns an incorrect answer:

> " #include <iostream> using namespace std; int main() { int dogs = 2; int cats = 3; int fish = 2 * (dogs + cats); int totalPets = dogs + cats + fish; cout << \"Ed has \" << totalPets << \" pets\" << endl; return 0; }\nEd has 11 pets\n\n(2 + 3 + 2*(2+3)) = 11\n\nExplanation:\n\n* Dogs: 2\n* Cats: 3\n* Fish: 2 * (2 + 3) = 2 * 5 = 10\n\nTotal pets: 2 + 3 + 10 = 11\n\n#### Explanation:\n\nLet's add up the number of pets:\n\n* Dogs: 2\n* Cats: 3\n* Fish: 2 * (2 + 3) = 2 * 5 = 10\n\nTotal pets: 2 + 3 + 10 = 11\n\n#### Explanation:\n\nLet's add up the number of pets:\n\n* Dogs: 2\n* Cats: 3\n* Fish: 2 * (2 + 3) = 2 * 5 = 10\n\nTotal pets: 2 + 3"


2. If we run LoRA inference with `llama3-8b-instruct-lora_vnemo-math-v1` and ask the same question: 

In [ ]:
!curl -X 'POST' \
  'http://0.0.0.0:8000/v1/completions' \
  -H 'accept: application/json' \
  -H 'Content-Type: application/json' \
  -d '{  \
        "model": "llama3-8b-instruct-lora_vnemo-math-v1",  \
        "prompt": "Ed has 2 dogs, 3 cats and twice as many fish as cats and dogs combined. How many pets does Ed have in total?",  \
        "max_tokens": 256  \
      }' | jq

We get the correct answer and much better step-by-step reasoning: 

> " First find the total number of cats and dogs: 2 + 3 = <<2+3=5>>5\nThen multiply this number by 2 to find how many fish Ed has: 5 * 2 = <<5*2=10>>10 fish\nNow add Ed's cats, dogs, and fish to find the total number of pets Ed has: 10 + 5 = <<10+5=15>>15 pets\n#### 15"

Once this has been completed, we can stop and remove the NIM container.

### **Optional** Stop and Remove the NIM Container

In [ ]:
# Check which GPU you have and how much memory we have to work with.
!nvidia-smi

In [ ]:
# Stop all docker containers (except nginx server)
!docker ps --format '{{.Names}} {{.ID}}' | grep -v "nginx" | awk '{print $2}' | xargs -r docker kill

---
## APPENDIX

### Other Evaluation Metrics for datasets for *summarization*:

### ROUGE and BLEU scores - Use on a Summarization Dataset

For summarization tasks, we can use other evaluation metrics to understand the real world behavior of the model. For our summarization use case, we can use ROUGE and BLEU scores for testing model performance.


You can save the following script as `rouge_bleu.py`. Please make sure to update the `TEST_PREDICTION_FILE` variable to appropriate jsonl file. 

```bash
cat > rouge_bleu.py <<"EOF"
import evaluate
import json

OUTPUT_FILE = "./results/peft_results_test_att_dataset_inputs_preds_labels.jsonl"

# Load the jsonl file
jsonl_data = []
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    jsonl_data = [json.loads(line) for line in f]

# Define the predictions and reference sentences
predictions = [data['pred'] for data in jsonl_data]
references = [[data['label']] for data in jsonl_data]

# Load and compute the BLEU evaluation metric
bleu = evaluate.load("bleu")
bleu_results = bleu.compute(predictions=predictions, references=references)
print(bleu_results)

# Load and compute ROUGE evaluation metric
rouge = evaluate.load('rouge')
rouge_results = rouge.compute(predictions=predictions, references=references)
print(rouge_results)
EOF

python rouge_bleu.py
```

In [ ]:


!docker run --gpus '"device=0"' --shm-size=2g --net=host \
    --ulimit memlock=-1 --rm -it -v ${PWD}:/workspace -w /workspace -v ${PWD}/results:/results nvcr.io/nvidia/nemo:24.12 \
        python scripts/rouge_bleu.py

-----------
# EOF